# Notebook 02: BM25 Retrieval

**Goal:** Build an in-memory sparse retriever (`BM25Retriever`) over document chunks and evaluate candidate retrieval with exact vs. paraphrased keyword queries.

## Objectives:
1. Initialize `BM25Retriever` from document chunks created in Notebook 01
2. Execute exact keyword match queries and verify top candidate
3. Execute paraphrased queries and inspect top-10 scores
4. Verify zero-match handling and run automated verification assertions

In [7]:
# Cell 1: Import dependencies & load chunks from Notebook 01
import os
import sys
from typing import List, Dict, Any

project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from scripts.create_sample_pdf import create_sample_pdf
import pdfplumber
import re
import nltk
from nltk.tokenize import sent_tokenize

# Document processing helper from Notebook 01
def get_sample_chunks() -> List[Document]:
    pdf_path = os.path.join(project_root, "data", "harry_potter.pdf")
    if not os.path.exists(pdf_path):
        create_sample_pdf(pdf_path)
    
    raw_pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for idx, page in enumerate(pdf.pages):
            raw_pages.append(page.extract_text() or "")
            
    full_text = "\n\n".join(raw_pages)
    section_splits = re.split(r'(?m)^(?:(?=\d+\.\s+[A-Z]))', full_text)
    
    docs = []
    chunk_id = 0
    for split in section_splits:
        split_clean = split.strip()
        if not split_clean:
            continue
        lines = split_clean.split('\n')
        if re.match(r'^\d+\.\s+[A-Z]', lines[0].strip()):
            sec_title = lines[0].strip()
            body_text = "\n".join(lines[1:]).strip()
        else:
            sec_title = "Document Title & Preamble"
            body_text = split_clean
            
        paragraphs = [re.sub(r'\s+', ' ', p).strip() for p in body_text.split('\n\n') if p.strip()]
        for p_idx, para in enumerate(paragraphs):
            if not para.endswith(('.', '!', '?', ':', ';')):
                para += '.'
            chunk_id += 1
            docs.append(Document(
                page_content=para,
                metadata={
                    "chunk_id": f"chunk_{chunk_id:03d}",
                    "section": sec_title,
                    "paragraph_index": p_idx + 1
                }
            ))
    return docs

documents = get_sample_chunks()
print(f"Loaded {len(documents)} document chunks for BM25 retriever.")

Loaded 98 document chunks for BM25 retriever.


In [8]:
# Cell 2: Build BM25Retriever
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 10
print(f"BM25Retriever successfully initialized with k={bm25_retriever.k}.")

BM25Retriever successfully initialized with k=10.


In [13]:
# Cell 3: Query with Exact Keyword Match
exact_query = "who is serius black?"
exact_results = bm25_retriever.invoke(exact_query)
print(f"Query: '{exact_query}'")
print(f"Retrieved {len(exact_results)} candidates.")
print(f"Top 1 Result Chunk ID: {exact_results[0].metadata['chunk_id']}")
print(f"Top 1 Content Preview: {exact_results[0].page_content[:150]}...")

Query: 'who is serius black?'
Retrieved 10 candidates.
Top 1 Result Chunk ID: chunk_012
Top 1 Content Preview: his ear.) “Forgive me, but — I do not understand — why should we wait until the World Cup is over?” “Because, fool, at this very moment wizards are po...


In [11]:
# Cell 4: Query with Paraphrased Keywords
paraphrased_query = "who is serius black?"
para_results = bm25_retriever.invoke(paraphrased_query)
print(f"Query: '{paraphrased_query}'")
print(f"Retrieved {len(para_results)} candidates.")
for i, res in enumerate(para_results[:3]):
    print(f"Rank {i+1} | {res.metadata['chunk_id']} | {res.page_content[:100]}...")

Query: 'who is serius black?'
Retrieved 10 candidates.
Rank 1 | chunk_012 | his ear.) “Forgive me, but — I do not understand — why should we wait until the World Cup is over?” ...
Rank 2 | chunk_013 | most effectively — I could be back here in as little as two days with a suitable person —” “I could ...
Rank 3 | chunk_014 | thirteen years. A few more months will make no difference. As for the protection surrounding the boy...


In [15]:
# Cell 5: Query with Zero Match Keywords
zero_match_query = "is there james bond in this document?"
zero_results = bm25_retriever.invoke(zero_match_query)
print(f"Zero-match Query: '{zero_match_query}'")
print(f"Retrieved Candidate Count: {len(zero_results)} (Handled cleanly without error)")

Zero-match Query: 'is there james bond in this document?'
Retrieved Candidate Count: 10 (Handled cleanly without error)


In [17]:
# Cell 6: Verification Assertions for Custom PDF
print("Running verification assertions on BM25 Retrieval...")

assert len(exact_results) > 0, "Verification Failed: Query returned 0 results!"
assert isinstance(exact_results, list), "Verification Failed: Output is not a list!"

print("✅ ALL VERIFICATION TESTS PASSED FOR HARRY POTTER RETRIEVAL!")


Running verification assertions on BM25 Retrieval...
✅ ALL VERIFICATION TESTS PASSED FOR HARRY POTTER RETRIEVAL!
